# Percobaan 6 - Reconstructed Historical Features + CatBoost

Notebook ini mencoba Jalur A yang lebih serius: menambah sinyal yang hilang di `test.csv`.

Yang direconstruct:
- recent form: points/goals/conceded/GD/win rate,
- H2H,
- days since last match,
- Elo sederhana,
- latest rank fallback dari history train,
- CatBoost ensemble,
- blend search,
- outcome-aware post-processing.

Prinsip penting: validation dan test tidak boleh memakai target mereka sendiri untuk membangun fitur outcome-dependent.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict, deque
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

## 1. Path dan konfigurasi

Default `TASK_TYPE='CPU'` supaya stabil. Kalau CatBoost GPU di kernel `py_gpu_ready` aman, boleh ganti ke `GPU`.

In [2]:
BASE_PATH = Path.home() / "Downloads" / "Gammafest"
DATA_PATH = BASE_PATH / "dataset"
OUTPUT_DIR = BASE_PATH / "experiments" / "percobaan 6 - reconstructed features"

TRAIN_PATH = DATA_PATH / "train.csv"
TEST_PATH = DATA_PATH / "test.csv"
SAMPLE_PATH = DATA_PATH / "sample submission.csv"

SUBMISSION_PATH = OUTPUT_DIR / "submission_reconstructed_catboost_jalur_a.csv"
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / "submission_reconstructed_catboost_roundclip.csv"
VALID_REPORT_PATH = OUTPUT_DIR / "validation_reconstructed_report.csv"
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / "postprocess_reconstructed_report.csv"

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = "CPU"
MAX_SCORE = 6
N_RANDOM_BLENDS = 2000

ELO_INIT = 1500.0
ELO_K_DEFAULT = 20
ELO_K_IMPORTANT = 40
IMPORTANT_TOURNAMENTS = {
    "FIFA World Cup", "AFC Asian Cup", "AFC Championship", "UEFA Euro",
    "Africa Cup of Nations", "African Cup of Nations", "Copa America", "Copa América",
    "Gold Cup", "CONCACAF Gold Cup"
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 6 - reconstructed features


## 2. Load data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

train_raw["date_dt"] = pd.to_datetime(train_raw["date"], errors="coerce")
test_raw["date_dt"] = pd.to_datetime(test_raw["date"], errors="coerce")

print("train:", train_raw.shape, train_raw["date_dt"].min(), "->", train_raw["date_dt"].max())
print("test :", test_raw.shape, test_raw["date_dt"].min(), "->", test_raw["date_dt"].max())
print("sample:", sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 48) 1872-11-30 00:00:00 -> 2011-08-04 00:00:00
test : (42422, 21) 2011-08-06 00:00:00 -> 2026-03-31 00:00:00
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169,2011-08-06
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169,2011-08-06


## 3. Static features, temporal split, dan AW-MAE constants

In [4]:
CAT_COLS = [
    "gender", "team", "opponent", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]

STATIC_NUM_COLS = [
    "is_home", "neutral",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp", "temperature_venue",
]

DATE_FEATURES = ["year", "month", "dayofweek"]

TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Championship": 1.80,
    "AFC Asian Cup": 1.80,
    "UEFA Euro": 1.80,
    "Copa America": 1.80,
    "Copa América": 1.80,
    "Africa Cup of Nations": 1.80,
    "African Cup of Nations": 1.80,
    "Gold Cup": 1.75,
    "CONCACAF Gold Cup": 1.75,
    "FIFA World Cup qualification": 1.50,
    "UEFA Euro qualification": 1.40,
    "AFC Asian Cup qualification": 1.40,
    "Friendly": 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def add_basic_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["date"], errors="coerce")
    df["date_dt"] = dt
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["tournament_weight"] = df["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT)
    return df


match_dates = train_raw.groupby("match_id")["date_dt"].min().sort_values()
split_idx = int(len(match_dates) * (1 - VALID_FRAC))
train_match_ids = set(match_dates.index[:split_idx])
val_match_ids = set(match_dates.index[split_idx:])

tr_raw = train_raw[train_raw["match_id"].isin(train_match_ids)].copy()
val_raw = train_raw[train_raw["match_id"].isin(val_match_ids)].copy()

print("Train fold rows:", tr_raw.shape, tr_raw["date_dt"].min(), "->", tr_raw["date_dt"].max())
print("Valid fold rows:", val_raw.shape, val_raw["date_dt"].min(), "->", val_raw["date_dt"].max())
print("Train matches:", len(train_match_ids), "Valid matches:", len(val_match_ids))

Train fold rows: (63016, 48) 1872-11-30 00:00:00 -> 2005-01-30 00:00:00
Valid fold rows: (15756, 48) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00
Train matches: 31508 Valid matches: 7878


## 4. Forward-only reconstruction

`can_update_outcome=True` hanya untuk history train yang boleh memperbarui state. Validation/test tidak memperbarui form, H2H, atau Elo dengan target mereka.
Schedule tetap memperbarui last-match date karena tanggal pertandingan memang diketahui.

In [5]:
def points_from_score(gf, ga):
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0


def elo_k(tournament):
    return ELO_K_IMPORTANT if tournament in IMPORTANT_TOURNAMENTS else ELO_K_DEFAULT


def summarize_form(history):
    if len(history) == 0:
        return {
            "points_last5": np.nan,
            "points_last10": np.nan,
            "gd_last5": np.nan,
            "avg_goals_last5": np.nan,
            "avg_conceded_last5": np.nan,
            "win_rate_last10": np.nan,
            "matches_known": 0,
        }
    last5 = list(history)[-5:]
    last10 = list(history)[-10:]
    return {
        "points_last5": float(sum(x["points"] for x in last5)),
        "points_last10": float(sum(x["points"] for x in last10)),
        "gd_last5": float(sum(x["gd"] for x in last5)),
        "avg_goals_last5": float(np.mean([x["gf"] for x in last5])),
        "avg_conceded_last5": float(np.mean([x["ga"] for x in last5])),
        "win_rate_last10": float(np.mean([x["points"] == 3 for x in last10])),
        "matches_known": int(len(history)),
    }


def build_reconstructed_features(input_df):
    df = add_basic_features(input_df).copy()
    df["_orig_order"] = np.arange(len(df))
    if "can_update_outcome" not in df.columns:
        df["can_update_outcome"] = df["team_goals"].notna() if "team_goals" in df.columns else False
    if "split_name" not in df.columns:
        df["split_name"] = "unknown"

    for col in ["team_goals", "opp_goals", "rank_team", "rank_opponent"]:
        if col not in df.columns:
            df[col] = np.nan

    df = df.sort_values(["date_dt", "match_id", "Id"]).reset_index(drop=True)

    form = defaultdict(lambda: deque(maxlen=50))
    h2h = defaultdict(lambda: deque(maxlen=20))
    elo = defaultdict(lambda: ELO_INIT)
    last_match_date = {}
    latest_rank = {}
    feature_rows = []

    for match_id, grp in df.groupby("match_id", sort=False):
        match_date = grp["date_dt"].iloc[0]

        for _, row in grp.iterrows():
            gender = row["gender"]
            team = row["team"]
            opp = row["opponent"]
            team_key = (gender, team)
            opp_key = (gender, opp)
            pair_key = (gender, team, opp)

            tf = summarize_form(form[team_key])
            of = summarize_form(form[opp_key])
            hhist = list(h2h[pair_key])[-5:]

            h2h_points = float(sum(x["points"] for x in hhist)) if hhist else np.nan
            h2h_gd = float(sum(x["gd"] for x in hhist)) if hhist else np.nan
            h2h_matches = int(len(hhist))

            team_last_date = last_match_date.get(team_key)
            opp_last_date = last_match_date.get(opp_key)
            days_team = (match_date - team_last_date).days if team_last_date is not None and pd.notna(match_date) else np.nan
            days_opp = (match_date - opp_last_date).days if opp_last_date is not None and pd.notna(match_date) else np.nan

            rank_team_latest = latest_rank.get(team_key, np.nan)
            rank_opp_latest = latest_rank.get(opp_key, np.nan)

            feature_rows.append({
                "_orig_order": row["_orig_order"],
                "team_points_last5_recon": tf["points_last5"],
                "opp_points_last5_recon": of["points_last5"],
                "points_last5_diff_recon": tf["points_last5"] - of["points_last5"] if pd.notna(tf["points_last5"]) and pd.notna(of["points_last5"]) else np.nan,
                "team_points_last10_recon": tf["points_last10"],
                "opp_points_last10_recon": of["points_last10"],
                "points_last10_diff_recon": tf["points_last10"] - of["points_last10"] if pd.notna(tf["points_last10"]) and pd.notna(of["points_last10"]) else np.nan,
                "team_gd_last5_recon": tf["gd_last5"],
                "opp_gd_last5_recon": of["gd_last5"],
                "gd_last5_diff_recon": tf["gd_last5"] - of["gd_last5"] if pd.notna(tf["gd_last5"]) and pd.notna(of["gd_last5"]) else np.nan,
                "team_avg_goals_last5_recon": tf["avg_goals_last5"],
                "team_avg_conceded_last5_recon": tf["avg_conceded_last5"],
                "opp_avg_goals_last5_recon": of["avg_goals_last5"],
                "opp_avg_conceded_last5_recon": of["avg_conceded_last5"],
                "avg_goals_diff_recon": tf["avg_goals_last5"] - of["avg_goals_last5"] if pd.notna(tf["avg_goals_last5"]) and pd.notna(of["avg_goals_last5"]) else np.nan,
                "avg_conceded_diff_recon": tf["avg_conceded_last5"] - of["avg_conceded_last5"] if pd.notna(tf["avg_conceded_last5"]) and pd.notna(of["avg_conceded_last5"]) else np.nan,
                "team_win_rate_last10_recon": tf["win_rate_last10"],
                "opp_win_rate_last10_recon": of["win_rate_last10"],
                "win_rate_diff_recon": tf["win_rate_last10"] - of["win_rate_last10"] if pd.notna(tf["win_rate_last10"]) and pd.notna(of["win_rate_last10"]) else np.nan,
                "h2h_points_last5_recon": h2h_points,
                "h2h_gd_last5_recon": h2h_gd,
                "h2h_matches_last5_recon": h2h_matches,
                "days_since_last_match_team_recon": days_team,
                "days_since_last_match_opp_recon": days_opp,
                "days_since_last_match_diff_recon": days_team - days_opp if pd.notna(days_team) and pd.notna(days_opp) else np.nan,
                "elo_team_recon": float(elo[team_key]),
                "elo_opponent_recon": float(elo[opp_key]),
                "elo_diff_recon": float(elo[team_key] - elo[opp_key]),
                "rank_team_latest_recon": rank_team_latest,
                "rank_opponent_latest_recon": rank_opp_latest,
                "rank_diff_latest_recon": rank_team_latest - rank_opp_latest if pd.notna(rank_team_latest) and pd.notna(rank_opp_latest) else np.nan,
                "rank_missing_team_recon": int(pd.isna(rank_team_latest)),
                "rank_missing_opp_recon": int(pd.isna(rank_opp_latest)),
                "team_matches_known_recon": tf["matches_known"],
                "opp_matches_known_recon": of["matches_known"],
                "matches_known_diff_recon": tf["matches_known"] - of["matches_known"],
            })

        for _, row in grp.iterrows():
            last_match_date[(row["gender"], row["team"])] = match_date

        can_update = bool(grp["can_update_outcome"].fillna(False).all())
        has_scores = grp["team_goals"].notna().all() and grp["opp_goals"].notna().all()
        if can_update and has_scores:
            for _, row in grp.iterrows():
                gender = row["gender"]
                team = row["team"]
                opp = row["opponent"]
                gf = float(row["team_goals"])
                ga = float(row["opp_goals"])
                pts = points_from_score(gf, ga)
                gd = gf - ga
                team_key = (gender, team)
                form[team_key].append({"points": pts, "gf": gf, "ga": ga, "gd": gd})
                h2h[(gender, team, opp)].append({"points": pts, "gd": gd})

                if pd.notna(row.get("rank_team", np.nan)):
                    latest_rank[team_key] = float(row["rank_team"])
                if pd.notna(row.get("rank_opponent", np.nan)):
                    latest_rank[(gender, opp)] = float(row["rank_opponent"])

            row = grp.iloc[0]
            gender = row["gender"]
            team_key = (gender, row["team"])
            opp_key = (gender, row["opponent"])
            e_team = elo[team_key]
            e_opp = elo[opp_key]
            expected_team = 1 / (1 + 10 ** ((e_opp - e_team) / 400))
            gf = float(row["team_goals"])
            ga = float(row["opp_goals"])
            actual_team = 1.0 if gf > ga else 0.5 if gf == ga else 0.0
            k = elo_k(row["tournament"])
            elo[team_key] = e_team + k * (actual_team - expected_team)
            elo[opp_key] = e_opp + k * ((1 - actual_team) - (1 - expected_team))

    feature_df = pd.DataFrame(feature_rows)
    return df.merge(feature_df, on="_orig_order", how="left").sort_values("_orig_order").reset_index(drop=True)

## 5. Build features untuk eval dan final test

In [6]:
tr_for_eval = tr_raw.copy()
tr_for_eval["split_name"] = "train_fold"
tr_for_eval["can_update_outcome"] = True

val_for_eval = val_raw.copy()
val_for_eval["split_name"] = "valid_fold"
val_for_eval["can_update_outcome"] = False

eval_input = pd.concat([tr_for_eval, val_for_eval], ignore_index=True, sort=False)
print("Building eval reconstructed features...")
eval_recon = build_reconstructed_features(eval_input)
tr_recon = eval_recon[eval_recon["split_name"] == "train_fold"].copy()
val_recon = eval_recon[eval_recon["split_name"] == "valid_fold"].copy()

full_train_for_final = train_raw.copy()
full_train_for_final["split_name"] = "full_train"
full_train_for_final["can_update_outcome"] = True

test_for_final = test_raw.copy()
test_for_final["split_name"] = "test"
test_for_final["can_update_outcome"] = False
test_for_final["team_goals"] = np.nan
test_for_final["opp_goals"] = np.nan

final_input = pd.concat([full_train_for_final, test_for_final], ignore_index=True, sort=False)
print("Building final reconstructed features...")
final_recon = build_reconstructed_features(final_input)
full_train_recon = final_recon[final_recon["split_name"] == "full_train"].copy()
test_recon = final_recon[final_recon["split_name"] == "test"].copy()

print("tr_recon:", tr_recon.shape)
print("val_recon:", val_recon.shape)
print("full_train_recon:", full_train_recon.shape)
print("test_recon:", test_recon.shape)

Building eval reconstructed features...
Building final reconstructed features...
tr_recon: (63016, 90)
val_recon: (15756, 90)
full_train_recon: (78772, 90)
test_recon: (42422, 90)


## 6. Sanity check dan preprocessing matrix

In [7]:
RECON_FEATURES = [
    "team_points_last5_recon", "opp_points_last5_recon", "points_last5_diff_recon",
    "team_points_last10_recon", "opp_points_last10_recon", "points_last10_diff_recon",
    "team_gd_last5_recon", "opp_gd_last5_recon", "gd_last5_diff_recon",
    "team_avg_goals_last5_recon", "team_avg_conceded_last5_recon",
    "opp_avg_goals_last5_recon", "opp_avg_conceded_last5_recon",
    "avg_goals_diff_recon", "avg_conceded_diff_recon",
    "team_win_rate_last10_recon", "opp_win_rate_last10_recon", "win_rate_diff_recon",
    "h2h_points_last5_recon", "h2h_gd_last5_recon", "h2h_matches_last5_recon",
    "days_since_last_match_team_recon", "days_since_last_match_opp_recon", "days_since_last_match_diff_recon",
    "elo_team_recon", "elo_opponent_recon", "elo_diff_recon",
    "rank_team_latest_recon", "rank_opponent_latest_recon", "rank_diff_latest_recon",
    "rank_missing_team_recon", "rank_missing_opp_recon",
    "team_matches_known_recon", "opp_matches_known_recon", "matches_known_diff_recon",
]

compare_pairs = [
    ("team_points_last5", "team_points_last5_recon"),
    ("opp_points_last5", "opp_points_last5_recon"),
    ("team_points_last10", "team_points_last10_recon"),
    ("opp_points_last10", "opp_points_last10_recon"),
    ("team_gd_last5", "team_gd_last5_recon"),
    ("opp_gd_last5", "opp_gd_last5_recon"),
    ("team_avg_goals_last5", "team_avg_goals_last5_recon"),
    ("team_avg_conceded_last5", "team_avg_conceded_last5_recon"),
    ("opp_avg_goals_last5", "opp_avg_goals_last5_recon"),
    ("opp_avg_conceded_last5", "opp_avg_conceded_last5_recon"),
    ("team_win_rate_last10", "team_win_rate_last10_recon"),
    ("opp_win_rate_last10", "opp_win_rate_last10_recon"),
    ("h2h_points_last5", "h2h_points_last5_recon"),
    ("h2h_gd_last5", "h2h_gd_last5_recon"),
    ("elo_team", "elo_team_recon"),
    ("elo_opponent", "elo_opponent_recon"),
]

rows = []
for orig, recon in compare_pairs:
    if orig in full_train_recon.columns and recon in full_train_recon.columns:
        tmp = full_train_recon[[orig, recon]].dropna()
        corr = tmp[orig].corr(tmp[recon]) if len(tmp) > 2 else np.nan
        rows.append({"original": orig, "reconstructed": recon, "non_null_pairs": len(tmp), "corr": corr})
compare_df = pd.DataFrame(rows).sort_values("corr")
display(compare_df)

BASE_NUM_COLS = STATIC_NUM_COLS + DATE_FEATURES + ["tournament_weight"] + RECON_FEATURES


def finalize_feature_frames(train_df, other_df, fit_name="train"):
    train = train_df.copy()
    other = other_df.copy()

    for df in [train, other]:
        for col in CAT_COLS:
            df[col] = df[col].fillna("Unknown").astype(str)
        for col in BASE_NUM_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in BASE_NUM_COLS:
        flag_col = f"{col}_missing"
        train[flag_col] = train[col].isna().astype(int)
        other[flag_col] = other[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[BASE_NUM_COLS].median(numeric_only=True)
    for col in BASE_NUM_COLS:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        other[col] = other[col].fillna(fill_value)

    feature_cols = CAT_COLS + BASE_NUM_COLS + missing_flag_cols
    cat_feature_indices = [feature_cols.index(c) for c in CAT_COLS]
    print(f"{fit_name}: features={len(feature_cols)}, missing train={train[feature_cols].isna().sum().sum()}, missing other={other[feature_cols].isna().sum().sum()}")
    return train, other, feature_cols, cat_feature_indices


tr_model, val_model, FEATURE_COLS, CAT_FEATURE_INDICES = finalize_feature_frames(tr_recon, val_recon, fit_name="eval")
full_train_model, test_model, FINAL_FEATURE_COLS, FINAL_CAT_FEATURE_INDICES = finalize_feature_frames(full_train_recon, test_recon, fit_name="final")

assert FEATURE_COLS == FINAL_FEATURE_COLS
assert CAT_FEATURE_INDICES == FINAL_CAT_FEATURE_INDICES

X_tr = tr_model[FEATURE_COLS]
X_val = val_model[FEATURE_COLS]
y_tr_team = tr_model["team_goals"]
y_tr_opp = tr_model["opp_goals"]
y_val_team = val_model["team_goals"]
y_val_opp = val_model["opp_goals"]

X_full = full_train_model[FEATURE_COLS]
y_full_team = full_train_model["team_goals"]
y_full_opp = full_train_model["opp_goals"]
X_test = test_model[FEATURE_COLS]

print("X_tr:", X_tr.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

,original,reconstructed,non_null_pairs,corr
8,opp_avg_goals_last5,opp_avg_goals_last5_recon,78289,0.800315
5,opp_gd_last5,opp_gd_last5_recon,78289,0.811221
9,opp_avg_conceded_last5,opp_avg_conceded_last5_recon,78289,0.824315
4,team_gd_last5,team_gd_last5_recon,78289,0.839995
1,opp_points_last5,opp_points_last5_recon,78289,0.841340
6,team_avg_goals_last5,team_avg_goals_last5_recon,78289,0.842844
7,team_avg_conceded_last5,team_avg_conceded_last5_recon,78289,0.843728
11,opp_win_rate_last10,opp_win_rate_last10_recon,78289,0.868597
3,opp_points_last10,opp_points_last10_recon,78289,0.875067
0,team_points_last5,team_points_last5_recon,78289,0.890021


eval: features=105, missing train=0, missing other=0
final: features=105, missing train=0, missing other=0
X_tr: (63016, 105) X_val: (15756, 105) X_test: (42422, 105)


## 7. AW-MAE dan post-processing helpers

In [8]:
def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    return np.clip(team, 0, max_score), np.clip(opp, 0, max_score)


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true["team_goals"].to_numpy()
    true_opp = df_true["opp_goals"].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()
    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        "AW-MAE": score,
        "MAE_raw_component": mae.mean(),
        "exact_acc": exact.mean(),
        "outcome_acc": outcome.mean(),
        "goal_diff_acc": gd.mean(),
    }
    return score, diag


def evaluate_raw_predictions(name, df_true, team_raw, opp_raw, max_score=MAX_SCORE):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    return {"model": name, **diag}

## 8. Train CatBoost ensemble

In [9]:
CATBOOST_CONFIGS = [
    {
        "name": "recon_cat_mae_d7_seed42",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1600, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "recon_cat_mae_d6_seed7",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1800, learning_rate=0.035, depth=6, l2_leaf_reg=9, random_strength=1.6, bagging_temperature=0.8, random_seed=7),
    },
    {
        "name": "recon_cat_mae_d8_seed99",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1400, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.3, random_seed=99),
    },
    {
        "name": "recon_cat_rmse_d7_seed123",
        "params": dict(loss_function="RMSE", eval_metric="MAE", iterations=1500, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

val_pred_bank = {}
trained_val_models = {}
reports = []

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=160)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=160)

    pred_t = np.clip(mt.predict(X_val), 0, None)
    pred_o = np.clip(mo.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (mt, mo)

    report = evaluate_raw_predictions(name, val_model, pred_t, pred_o, max_score=MAX_SCORE)
    report["best_iter_team"] = mt.best_iteration_
    report["best_iter_opp"] = mo.best_iteration_
    reports.append(report)
    print(report)

print("Training minutes:", (time.time() - start) / 60)
reports_df = pd.DataFrame(reports).sort_values("AW-MAE")
reports_df.to_csv(VALID_REPORT_PATH, index=False)
display(reports_df)
print("Saved:", VALID_REPORT_PATH)


Training recon_cat_mae_d7_seed42
0:	learn: 1.1699915	test: 1.1291791	best: 1.1291791 (0)	total: 128ms	remaining: 3m 24s
150:	learn: 1.0206560	test: 1.0281372	best: 1.0281372 (150)	total: 8.42s	remaining: 1m 20s
300:	learn: 1.0025163	test: 1.0263747	best: 1.0263747 (300)	total: 17.6s	remaining: 1m 16s
450:	learn: 0.9851808	test: 1.0253633	best: 1.0253570 (449)	total: 25.9s	remaining: 1m 5s
600:	learn: 0.9719029	test: 1.0254558	best: 1.0253384 (462)	total: 35.2s	remaining: 58.5s
750:	learn: 0.9601068	test: 1.0252252	best: 1.0249983 (696)	total: 48.5s	remaining: 54.9s
Stopped by overfitting detector  (160 iterations wait)

bestTest = 1.024998299
bestIteration = 696

Shrink model to first 697 iterations.
0:	learn: 1.1698202	test: 1.1288822	best: 1.1288822 (0)	total: 81ms	remaining: 2m 9s
150:	learn: 1.0215375	test: 1.0276397	best: 1.0276210 (147)	total: 13.6s	remaining: 2m 10s
300:	learn: 1.0037314	test: 1.0256635	best: 1.0256635 (300)	total: 26.3s	remaining: 1m 53s
450:	learn: 0.9860520	

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,recon_cat_rmse_d7_seed123,3.079975,1.034653,0.096344,0.568291,0.230388,1490,440
1,recon_cat_mae_d6_seed7,3.090248,0.995462,0.109101,0.521706,0.228675,1004,1420
2,recon_cat_mae_d8_seed99,3.090370,0.995430,0.108530,0.520310,0.228294,646,561
0,recon_cat_mae_d7_seed42,3.094304,0.998255,0.108974,0.520817,0.229183,696,795


Saved: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 6 - reconstructed features\validation_reconstructed_report.csv


## 9. Blend search

In [10]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t_int, pred_o_int)
    return score, diag


blend_rows = []
best = {"score": np.inf, "weights": None, "name": None, "diag": None}

for i, name in enumerate(model_names):
    w = np.zeros(len(model_names)); w[i] = 1
    score, diag = score_blend(w)
    blend_rows.append({"blend": f"single_{name}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"single_{name}", "diag": diag}

w = np.ones(len(model_names)) / len(model_names)
score, diag = score_blend(w)
blend_rows.append({"blend": "equal_average", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "equal_average", "diag": diag}

single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag = score_blend(w)
blend_rows.append({"blend": "inverse_awmae", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "inverse_awmae", "diag": diag}

rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag = score_blend(w)
    if k < 25 or score < best["score"]:
        blend_rows.append({"blend": f"random_{k}_a{alpha}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"random_{k}_a{alpha}", "diag": diag}

blend_df = pd.DataFrame(blend_rows).sort_values("AW-MAE").reset_index(drop=True)
display(blend_df.head(20))

print("Best blend:", best["name"])
print("Best AW-MAE:", best["score"])
print("Diagnostics:", best["diag"])
print("Weights:")
for name, weight in sorted(zip(model_names, best["weights"]), key=lambda x: -x[1]):
    print(f"  {name:<32} {weight:.5f}")

,blend,AW-MAE,w_recon_cat_mae_d7_seed42,w_recon_cat_mae_d6_seed7,w_recon_cat_mae_d8_seed99,w_recon_cat_rmse_d7_seed123
0,random_52_a0.25,3.042618,0.079750,0.398835,0.014855,0.506560
1,random_31_a2.0,3.046026,0.110881,0.151535,0.142076,0.595508
2,random_5_a0.5,3.050627,0.026982,0.100146,0.299397,0.573474
3,random_8_a0.25,3.058485,0.000823,0.429836,0.219821,0.349521
4,random_9_a0.5,3.062219,0.197650,0.000592,0.028711,0.773046
5,random_1_a0.5,3.063340,0.017968,0.150514,0.490905,0.340613
6,random_22_a1.0,3.066909,0.398764,0.116891,0.179967,0.304377
7,random_23_a2.0,3.068883,0.151081,0.139291,0.410416,0.299212
8,random_10_a1.0,3.070333,0.011439,0.212649,0.587119,0.188793
9,random_19_a2.0,3.071356,0.037639,0.199239,0.549128,0.213994


Best blend: random_52_a0.25
Best AW-MAE: 3.042617849049221
Diagnostics: {'AW-MAE': np.float64(3.042617849049221), 'MAE_raw_component': np.float64(1.004823559279005), 'exact_acc': np.float64(0.10389692815435389), 'outcome_acc': np.float64(0.5467758314292968), 'goal_diff_acc': np.float64(0.23235592790048235)}
Weights:
  recon_cat_rmse_d7_seed123        0.50656
  recon_cat_mae_d6_seed7           0.39883
  recon_cat_mae_d7_seed42          0.07975
  recon_cat_mae_d8_seed99          0.01485


## 10. Max score dan outcome-aware post-processing

In [11]:
def build_score_pair_prior(df, max_score=6, smoothing=1.0):
    counts = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)
    team = np.clip(df["team_goals"].round().astype(int).to_numpy(), 0, max_score)
    opp = np.clip(df["opp_goals"].round().astype(int).to_numpy(), 0, max_score)
    for tg, og in zip(team, opp):
        counts[tg, og] += 1.0
    return counts / counts.sum()


def soft_outcome_from_raw(team_raw, opp_raw, draw_margin=0.20):
    diff = np.asarray(team_raw) - np.asarray(opp_raw)
    return np.where(diff > draw_margin, 1, np.where(diff < -draw_margin, -1, 0))


def outcome_aware_postprocess(team_raw, opp_raw, params, prior_matrix=None):
    max_score = int(params.get("max_score", 6))
    draw_margin = float(params.get("draw_margin", 0.20))
    outcome_weight = float(params.get("outcome_weight", 0.10))
    gd_weight = float(params.get("gd_weight", 0.05))
    prior_weight = float(params.get("prior_weight", 0.02))

    team_raw = np.asarray(team_raw, dtype=float)
    opp_raw = np.asarray(opp_raw, dtype=float)
    candidates = np.array([(tg, og) for tg in range(max_score + 1) for og in range(max_score + 1)], dtype=int)
    cand_team = candidates[:, 0]
    cand_opp = candidates[:, 1]
    cand_diff = cand_team - cand_opp
    cand_outcome = outcome_label(cand_team, cand_opp)

    base_cost = (np.abs(team_raw[:, None] - cand_team[None, :]) + np.abs(opp_raw[:, None] - cand_opp[None, :])) / 2
    gd_cost = np.abs((team_raw - opp_raw)[:, None] - cand_diff[None, :])
    raw_outcome = soft_outcome_from_raw(team_raw, opp_raw, draw_margin=draw_margin)
    outcome_cost = (raw_outcome[:, None] != cand_outcome[None, :]).astype(float)
    total_cost = base_cost + gd_weight * gd_cost + outcome_weight * outcome_cost

    if prior_matrix is not None and prior_weight > 0:
        prior = np.clip(prior_matrix[cand_team, cand_opp], 1e-12, None)
        total_cost = total_cost + prior_weight * (-np.log(prior))[None, :]

    best_idx = np.argmin(total_cost, axis=1)
    return cand_team[best_idx].astype(int), cand_opp[best_idx].astype(int)


blend_weights = best["weights"] / best["weights"].sum()
val_team_raw = np.average(team_matrix, axis=0, weights=blend_weights)
val_opp_raw = np.average(opp_matrix, axis=0, weights=blend_weights)

clip_rows = []
for max_score in range(4, 9):
    pred_t, pred_o = postprocess_round_clip(val_team_raw, val_opp_raw, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    clip_rows.append({"max_score": max_score, **diag})
clip_df = pd.DataFrame(clip_rows).sort_values("AW-MAE")
display(clip_df)

base_max_score = int(clip_df.iloc[0]["max_score"])
base_pred_t, base_pred_o = postprocess_round_clip(val_team_raw, val_opp_raw, max_score=base_max_score)
base_score, base_diag = compute_awmae(val_model, base_pred_t, base_pred_o)
print("Round+clip best:", base_score, "max_score:", base_max_score)

max_score_candidates = sorted(set([base_max_score, max(4, base_max_score - 1), min(8, base_max_score + 1)]))
draw_margins = [0.00, 0.10, 0.20, 0.30, 0.40]
outcome_weights = [0.00, 0.05, 0.10, 0.20, 0.35]
gd_weights = [0.00, 0.03, 0.06, 0.10]
prior_weights = [0.00, 0.01, 0.03, 0.06]

pp_rows = []
best_pp = {
    "score": base_score,
    "params": {"mode": "round_clip", "max_score": base_max_score, "draw_margin": None, "outcome_weight": 0.0, "gd_weight": 0.0, "prior_weight": 0.0},
    "diag": base_diag,
}

start = time.time()
for max_score in max_score_candidates:
    prior_matrix = build_score_pair_prior(tr_model, max_score=max_score, smoothing=1.0)
    for draw_margin in draw_margins:
        for outcome_weight in outcome_weights:
            for gd_weight in gd_weights:
                for prior_weight in prior_weights:
                    params = {"mode": "outcome_aware", "max_score": max_score, "draw_margin": draw_margin, "outcome_weight": outcome_weight, "gd_weight": gd_weight, "prior_weight": prior_weight}
                    pred_t, pred_o = outcome_aware_postprocess(val_team_raw, val_opp_raw, params, prior_matrix=prior_matrix)
                    score, diag = compute_awmae(val_model, pred_t, pred_o)
                    pp_rows.append({**params, **diag})
                    if score < best_pp["score"]:
                        best_pp = {"score": score, "params": params.copy(), "diag": diag.copy()}

pp_df = pd.DataFrame(pp_rows).sort_values("AW-MAE").reset_index(drop=True)
pp_df.to_csv(POSTPROCESS_REPORT_PATH, index=False)
BEST_PP_PARAMS = best_pp["params"]
BEST_PP_SCORE = best_pp["score"]
BEST_MAX_SCORE = int(BEST_PP_PARAMS["max_score"])

print("Postprocess search minutes:", (time.time() - start) / 60)
print("Best postprocess score:", BEST_PP_SCORE)
print("Best postprocess params:", BEST_PP_PARAMS)
print("Best diagnostics:", best_pp["diag"])
display(pp_df.head(20))

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,4,3.033372,1.002856,0.104722,0.546776,0.234006
1,5,3.039134,1.003903,0.102691,0.546776,0.231340
2,6,3.042618,1.004824,0.103897,0.546776,0.232356
3,7,3.047768,1.006632,0.102945,0.546776,0.231277
4,8,3.052078,1.008187,0.102818,0.546776,0.231150


Round+clip best: 3.0333723912032 max_score: 4
Postprocess search minutes: 0.20406407515207928
Best postprocess score: 2.9639597275550287
Best postprocess params: {'mode': 'outcome_aware', 'max_score': 4, 'draw_margin': 0.1, 'outcome_weight': 0.35, 'gd_weight': 0.06, 'prior_weight': 0.06}
Best diagnostics: {'AW-MAE': np.float64(2.9639597275550287), 'MAE_raw_component': np.float64(1.0157400355420156), 'exact_acc': np.float64(0.10066006600660066), 'outcome_acc': np.float64(0.5942498095963442), 'goal_diff_acc': np.float64(0.22594567149022596)}


,mode,max_score,draw_margin,outcome_weight,gd_weight,prior_weight,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,outcome_aware,4,0.1,0.35,0.06,0.06,2.963960,1.015740,0.100660,0.594250,0.225946
1,outcome_aware,4,0.1,0.35,0.00,0.06,2.964195,1.018088,0.101041,0.596852,0.226580
2,outcome_aware,4,0.1,0.35,0.03,0.06,2.964680,1.017009,0.100724,0.594821,0.226200
3,outcome_aware,4,0.0,0.20,0.03,0.06,2.964843,1.006823,0.104214,0.580795,0.234514
4,outcome_aware,4,0.2,0.35,0.00,0.06,2.964859,1.014629,0.103008,0.592854,0.229246
5,outcome_aware,4,0.0,0.20,0.06,0.06,2.965364,1.006823,0.103262,0.580096,0.233816
6,outcome_aware,4,0.0,0.35,0.06,0.06,2.965402,1.017200,0.099962,0.595392,0.224930
7,outcome_aware,4,0.1,0.35,0.10,0.06,2.966125,1.014407,0.101358,0.592409,0.226707
8,outcome_aware,4,0.2,0.35,0.06,0.06,2.966503,1.013899,0.101549,0.591203,0.227786
9,outcome_aware,4,0.0,0.35,0.03,0.06,2.966609,1.018913,0.099962,0.596408,0.225057


## 11. Train final models dan generate submission

In [12]:
final_pred_bank = {}
final_models = {}

for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 350)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)

    final_pred_bank[name] = (np.clip(mt.predict(X_test), 0, None), np.clip(mo.predict(X_test), 0, None))
    final_models[name] = (mt, mo)

test_team_matrix = np.vstack([final_pred_bank[name][0] for name in model_names])
test_opp_matrix = np.vstack([final_pred_bank[name][1] for name in model_names])
weights = best["weights"] / best["weights"].sum()
test_team_raw = np.average(test_team_matrix, axis=0, weights=weights)
test_opp_raw = np.average(test_opp_matrix, axis=0, weights=weights)

round_team, round_opp = postprocess_round_clip(test_team_raw, test_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[["Id"]].copy()
submission_round["team_goals"] = round_team
submission_round["opp_goals"] = round_opp
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

if BEST_PP_PARAMS.get("mode") == "outcome_aware":
    prior_matrix = build_score_pair_prior(full_train_model, max_score=int(BEST_PP_PARAMS["max_score"]), smoothing=1.0)
    final_team, final_opp = outcome_aware_postprocess(test_team_raw, test_opp_raw, BEST_PP_PARAMS, prior_matrix=prior_matrix)
    active_mode = "outcome_aware"
else:
    final_team, final_opp = round_team, round_opp
    active_mode = "round_clip"

submission = sample[["Id"]].copy()
submission["team_goals"] = final_team
submission["opp_goals"] = final_opp
assert submission.shape == sample.shape
assert submission["Id"].equals(sample["Id"])
submission.to_csv(SUBMISSION_PATH, index=False)

changed_rows = ((submission["team_goals"] != submission_round["team_goals"]) | (submission["opp_goals"] != submission_round["opp_goals"])).sum()

print("Saved submission:", SUBMISSION_PATH)
print("Saved round+clip backup:", SUBMISSION_ROUNDCLIP_PATH)
print("Active mode:", active_mode)
print("Validation best blend AW-MAE:", best["score"])
print("Validation best postprocess AW-MAE:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Changed rows vs round+clip:", changed_rows)
print("Shape:", submission.shape)

display(submission.head())
display(submission[["team_goals", "opp_goals"]].describe())

print("\ndistribusi skor")
print("\nteam_goals")
print(submission["team_goals"].value_counts())
print("\nopp_goals")
print(submission["opp_goals"].value_counts())

print("\ndistribusi pasangan skor")
display(submission[["team_goals", "opp_goals"]].value_counts().head(30).to_frame("count"))


Final training recon_cat_mae_d7_seed42 iterations: 935
0:	learn: 1.1613448	total: 62.8ms	remaining: 58.7s
150:	learn: 1.0114035	total: 8.97s	remaining: 46.6s
300:	learn: 0.9953008	total: 17.8s	remaining: 37.4s
450:	learn: 0.9806768	total: 26.6s	remaining: 28.5s
600:	learn: 0.9687846	total: 35.4s	remaining: 19.7s
750:	learn: 0.9594888	total: 44.1s	remaining: 10.8s
900:	learn: 0.9513416	total: 52.9s	remaining: 2s
934:	learn: 0.9497385	total: 54.9s	remaining: 0us
0:	learn: 1.1604942	total: 61.9ms	remaining: 57.8s
150:	learn: 1.0129106	total: 8.74s	remaining: 45.4s
300:	learn: 0.9970300	total: 17.5s	remaining: 36.9s
450:	learn: 0.9820707	total: 26.3s	remaining: 28.3s
600:	learn: 0.9708410	total: 35.1s	remaining: 19.5s
750:	learn: 0.9608396	total: 43.8s	remaining: 10.7s
900:	learn: 0.9524748	total: 52.5s	remaining: 1.98s
934:	learn: 0.9507202	total: 54.5s	remaining: 0us

Final training recon_cat_mae_d6_seed7 iterations: 1560
0:	learn: 1.1623231	total: 53.4ms	remaining: 1m 23s
150:	learn: 1

,Id,team_goals,opp_goals
0,M034984_Seychelles,2,1
1,M034984_Mauritius,1,2
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,2,1


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.469379,1.477276
std,1.019248,1.017603
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,4.000000,4.000000



distribusi skor

team_goals
team_goals
1    18147
2    12463
0     6215
3     3127
4     2470
Name: count, dtype: int64

opp_goals
opp_goals
1    18124
2    12616
0     6076
3     3111
4     2495
Name: count, dtype: int64

distribusi pasangan skor


,,count
team_goals,opp_goals,
1,2,10406
2,1,10338
1,1,5694
0,4,2052
4,0,2024
0,2,1915
2,0,1853
0,3,1784
3,0,1752


## 12. Notes

Kalau hasil Kaggle belum naik:
1. validation ini lebih realistis daripada notebook lama, jadi local score bisa terlihat lebih buruk,
2. fitur form untuk test masih frozen dari train karena target test tidak diketahui,
3. next upgrade pure ML: pseudo-sequential update test memakai prediksi tahap pertama, classifier W/D/L, dan calibration per gender/tournament.